# Code to reproduce the results in the report

In [1]:
import pandas as pd
import ast 

from discovery_child_development import PROJECT_DIR, S3_BUCKET, logging
from discovery_child_development.analysis.initial_results import utils
from nesta_ds_utils.loading_saving import S3

from discovery_child_development.utils import analysis_utils as au
from discovery_child_development.utils import plotting_utils as pu
from discovery_child_development.utils import chart_trends

# Remove altair warning
import altair as alt
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

S3_OUTPUTS_DIR = '2024-07-iss-child-development/outputs/'

2024-07-26 09:36:10,504 - botocore.credentials - INFO - Found credentials in environment variables.
2024-07-26 09:36:12,269 - datasets - INFO - PyTorch version 2.4.0 available.


/opt/homebrew/Caskroom/miniconda/base/envs/discovery_child_development/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
TECH = 'Technology'

# Taxonomy dataframe
topics_df = utils.load_topic_data()

# List all tech major categories
tech_subtypes = set(topics_df.query("type == @TECH").subtype.unique())
print(tech_subtypes)

{'AI', 'Mobile', 'Internet', 'Immersive tech'}


## Helper functions

In [3]:
# Relevant document ids for time series
def get_tech_ids(data_exploded_df):
    """Get relevant document ids for time series/growth estimations"""
    return (
        data_exploded_df
        .query("type == @TECH")
        .query("year >= 2013")
        .drop_duplicates('id')
        .id.to_list()
)

def get_tech_ids_5y(data_exploded_df: pd.DataFrame) -> pd.DataFrame:
    """Get relevant document ids for 2019-2023 stats"""
    return (
        data_exploded_df
        .query("type == @TECH")
        .query("year >= 2019")
        .drop_duplicates('id')
        .id.to_list()
    )

def report_magnitude_growth(magnitude_growth_df: pd.DataFrame) -> None:
    """Report magnitude and growth"""
    # Smoothed growth in 2019-2023
    growth = magnitude_growth_df.growth.iloc[0]
    # Total funding in 2019-2023 (in millions)
    magnitude = magnitude_growth_df.magnitude.iloc[0] * 5 

    logging.info(f"Growth in 2019-2023: {growth:.2f}%")
    logging.info(f"Total in 2019-2023: {magnitude:.2f}")

    return growth, magnitude


def get_tech_distribution(data_exploded_df: pd.DataFrame, tech_ids_5y, values: list, column: str='subtype') -> pd.DataFrame:
    """Get distribution of technology projects"""
    return utils.get_data_distribution(
        (
            data_exploded_df
            .query('id in @tech_ids_5y')
            .query("type == @TECH")
            .drop_duplicates(['id', column])
        ),
        column=column, 
        values=values,
    ) 


def get_tech_ts(data_exploded_df: pd.DataFrame, tech_ids, values: list, column: str='subtype') -> pd.DataFrame:
    """Get distribution of technology projects"""
    return utils.get_data_distribution(
        (
            data_exploded_df
            .query('id in @tech_ids')
            .query("type == @TECH")
            .drop_duplicates(['id', column])
        ),
        column=column, 
        values=values,
        ts = True,
    )  

def get_tech_trends(data_exploded_df: pd.DataFrame, tech_ids, tech_ids_5y, values: list, column='subtype') -> pd.DataFrame:
    tech_dist = get_tech_distribution(data_exploded_df, tech_ids_5y, values, column=column)
    tech_ts = get_tech_ts(data_exploded_df, tech_ids, values, column=column)
    _value = 'counts' if values[0] == 'id' else values[0]
    tech_magnitude_growth = utils.magnitude_and_growth(
        tech_ts, 
        column = column, 
        value=_value)
    tech_trends = (
        tech_dist.merge(
            tech_magnitude_growth, 
            on=column, 
            how='left')
    )
    return tech_trends

def get_application_trends(data_exploded_df: pd.DataFrame, tech_ids, tech_ids_5y, values: list, column:str = 'type', hide_categories: list = ['Technology', 'General']) -> pd.DataFrame:
    """Get application trends"""
    tech_applications_df = utils.get_data_distribution(
        data_exploded_df.query('id in @tech_ids_5y'), 
        column=column, 
        values=values,
    ) 
    trends_df = utils.get_data_magnitude_growth(
        data_exploded_df, ids=tech_ids, 
        column=column, 
        value=values[0])  
    return (
        tech_applications_df
        .merge(trends_df.drop('counts', axis=1), on='type')
        .query("type not in @hide_categories")
        ) 


def trends_chart(application_trends_df: pd.DataFrame) -> pd.DataFrame:
    return (
        alt.Chart(application_trends_df)
        .mark_point()
        .encode(
            x='magnitude:Q',
            y='growth:Q',
            color='type:N',
            tooltip=['type', 'magnitude', 'growth'],
        )
    )

## Research funding
- Fig 2: Growth and total early-years digital tech research funding in 2019-2023
- Breakdowns by funder type (top funder; proportion from Innovate UK)
- Fig 4: Proportion of early-years funding associated with digital technologies
- Fig 5: Proportion and growth of major digital tech categories in 2019-2023
- Fig 6: Proportion and growth of major application areas in 2019-2023
- Baseline funding growth across all sectors in 2019-2023


In [4]:
# Load the data
ukri_df = (
    S3.download_obj(
        bucket = S3_BUCKET,
        path_from = S3_OUTPUTS_DIR + 'data/data_ukri.csv',
        download_as='dataframe'
    )
    # Remove the instances tagged with expressive arts due to too much noise
    .query("topics != 'arts'")
)

# Explode by topics
ukri_exploded_df = utils.explode_data(ukri_df).query("topics != 'arts'")

# Get relevant document ids
ukri_tech_ids = get_tech_ids(ukri_exploded_df)
ukri_tech_ids_5y = get_tech_ids_5y(ukri_exploded_df)


In [5]:
logging.info(f"Total number of UK research projects {len(ukri_df)}")

2024-07-26 09:36:13,194 - root - INFO - Total number of UK research projects 1085


### Growth and total early-years digital tech research funding

Figure 2

In [6]:
# Filter only technology projects
ukri_tech_type_df = (
    ukri_exploded_df
    .query("id in @ukri_tech_ids")
    .drop_duplicates(['id'])
)

ts_amounts_tech = utils.get_timeseries(ukri_tech_type_df, column='amount')
ts_counts_tech = utils.get_timeseries(ukri_tech_type_df, column='id')
utils.plot_quick_ts(ts_amounts_tech, 'amount')

alt.Chart(...)

In [7]:
# Get magnitude and growth
magnitude_growth_ukri = au.ts_magnitude_growth_(
    ts_amounts_tech,
    year_start = 2019,
    year_end = 2023
)

growth, magnitude = report_magnitude_growth(magnitude_growth_ukri)

2024-07-26 09:36:14,438 - root - INFO - Growth in 2019-2023: 165.74%
2024-07-26 09:36:14,439 - root - INFO - Total in 2019-2023: 58838.91


### Breakdowns by funder type
- Top funders
- Proportion from Innovate UK

In [8]:
# Check who are the top research funders for early-years digital tech
top_funders = (
    ukri_exploded_df
    .drop_duplicates(['id', 'lead_funder'])
    .query("id in @ukri_tech_ids_5y")    
    .groupby('lead_funder')
    .agg(amount=('amount', 'sum'))
    .sort_values('amount', ascending=False)
    .assign(proportion = lambda df: df.amount / df.amount.sum())
)
top_funders.head(5)

,amount,proportion
lead_funder,,
MRC,21617.556,0.367402
ESRC,5727.531,0.097343
Innovate UK,5584.073,0.094904
BBSRC,5425.603,0.092211
FLF,4700.880,0.079894


In [9]:
# Projects funded in the past five years
_funding_df = (
    ukri_exploded_df
    .query("id in @ukri_tech_ids_5y")
    .drop_duplicates('id')
)

# Get the total funding
funding_total = _funding_df.amount.sum()

# Get Innovate UK funding specifically
funding_innovate_uk = (
    _funding_df
    .query("lead_funder == 'Innovate UK'")
    .amount.sum())

# Calculate proportion of funding by Innovate UK
proportion_innovate_uk = funding_innovate_uk / funding_total

logging.info(f"Propotion of funding by Innovate UK: {proportion_innovate_uk:.2f}")

2024-07-26 09:36:18,942 - root - INFO - Propotion of funding by Innovate UK: 0.09


In [10]:
# Get funding excluding health-related projects (for reference)
health_ids = ukri_exploded_df.query("type == 'Health'").id.to_list()
funding_wout_health = _funding_df.query("id not in @health_ids").amount.sum()

# Calculate proportion of funding by Innovate UK excluding health
funding_innovate_uk_wout_health = (
    _funding_df
    .query("lead_funder == 'Innovate UK'")
    .query("id not in @health_ids")
    .amount.sum())

proportion_innovate_uk_wout_health = funding_innovate_uk_wout_health / funding_wout_health

logging.info(f"Propotion of funding by Innovate UK excluding health: {proportion_innovate_uk_wout_health:.2f}")

2024-07-26 09:36:21,712 - root - INFO - Propotion of funding by Innovate UK excluding health: 0.17


### Proportion of early-years funding associated with digital technologies

Figure 4

In [11]:
# Calculate the total early-years project funding
total_early_years_funding = (
    ukri_exploded_df
    .query("year >= 2019 and year <= 2023")
    .drop_duplicates('id')
    .amount.sum())

# Get the proportion of funding for early-years digital tech
proportion_early_years_tech_ukri = funding_total / total_early_years_funding
logging.info(f"Proportion of funding for early-years digital tech: {proportion_early_years_tech_ukri:.2f}")

2024-07-25 15:04:56,658 - root - INFO - Proportion of funding for early-years digital tech: 0.20


In [12]:
# Check Technology proportion using another approach
utils.get_data_distribution(
    ukri_exploded_df.query("year >= 2019 and year <= 2023"),
    column='type', 
    values=['id', 'amount']
)

,type,counts,counts_prop,amount,amount_prop
0,Biosciences,125,0.222,84393.908,0.288
1,Child care & preschool,23,0.041,16093.698,0.055
2,Development & learning,123,0.218,56456.251,0.193
3,General,348,0.618,195291.562,0.667
4,Health,362,0.643,209860.724,0.717
5,Parenting,20,0.036,8121.362,0.028
6,Society,133,0.236,68456.721,0.234
7,Technology,105,0.187,58838.909,0.201


### Digital tech trends in 2019-2023 (funding)

Figure 5

In [13]:
ukri_tech_trends = get_tech_trends(
    ukri_exploded_df, 
    ukri_tech_ids, 
    ukri_tech_ids_5y, 
    ['amount', 'id']
)

ukri_tech_trends

,subtype,amount,amount_prop,counts,counts_prop,type,magnitude,growth
0,AI,40910.811,0.695,62,0.59,Technology,8182.1622,147.968645
1,Immersive tech,7718.759,0.131,24,0.229,Technology,1543.7518,23.475780
2,Internet,5814.574,0.099,18,0.171,Technology,1162.9148,378.897783
3,Mobile,17869.267,0.304,23,0.219,Technology,3573.8534,87.113273


### Application area trends in 2019-2023 (funding)

Figure 6

In [14]:
ukri_application_trends = get_application_trends(
    ukri_exploded_df, 
    ukri_tech_ids, 
    ukri_tech_ids_5y, 
    ['amount', 'id']
)

ukri_application_trends


,type,amount,amount_prop,counts,counts_prop,magnitude,growth
0,Biosciences,13298.835,0.226,23,0.219,2659.767,65.806908
1,Child care & preschool,2590.215,0.044,8,0.076,518.043,299.955036
2,Development & learning,12640.91,0.215,31,0.295,2528.182,-51.255006
4,Health,40250.779,0.684,64,0.61,8050.1558,210.796765
5,Parenting,3059.329,0.052,9,0.086,611.8658,2541.467658
6,Society,16255.875,0.276,21,0.2,3251.175,148.675879


Trends information for the heat map.

Note that the trend typology is data-informed - we're guided by the results below - but we might also slighty adjust the final trends category in a few select cases.

In [15]:
chart_trends.estimate_trend_type(
    ukri_application_trends, 
    magnitude_column='magnitude', 
    growth_column='growth'
)[['type', 'magnitude', 'growth', 'trend_type_suggestion']]

,type,magnitude,growth,trend_type_suggestion
0,Biosciences,2659.767,65.806908,hot*
1,Child care & preschool,518.043,299.955036,emerging
2,Development & learning,2528.182,-51.255006,dormant*
4,Health,8050.1558,210.796765,hot
5,Parenting,611.8658,2541.467658,emerging
6,Society,3251.175,148.675879,hot


Designating Dev and learning as stabilising as the magnitue is in fact quite large relatively speaking

In [16]:
fig = trends_chart(
    ukri_application_trends
    .query("type != 'Biosciences'")
)
fig

alt.Chart(...)

### Baseline funding growth across all sectors

In [11]:
gtr_df = S3.download_obj(
    bucket = S3_BUCKET,
    path_from = S3_OUTPUTS_DIR + 'gtr_texts.csv',
    download_as='dataframe'
)

In [12]:
ukri_baseline_df = utils.get_baseline_ukri(gtr_df)

trends_baseline = au.ts_magnitude_growth_(
    ts_df = ukri_baseline_df,
    year_start = 2019,
    year_end = 2023  
)
ukri_baseline_magnitude = trends_baseline.loc['amount'].magnitude
ukri_baseline_growth = trends_baseline.loc['amount'].growth
logging.info(f"UKRI baseline growth: {ukri_baseline_growth:.2f}%")

2024-07-26 09:37:02,152 - root - INFO - UKRI baseline growth: -5.05%


In [28]:
_gtr_all_projects_df = (
    gtr_df
    .assign(year = lambda df: df.start.apply(lambda x: int(x[0:4])))
    .query("year >= 2019 and year <= 2023")
)

In [48]:
ukri_all_projects_funding = _gtr_all_projects_df.dropna(subset=["leadFunder"]).amount.sum()
ukri_innovate_uk_funding = _gtr_all_projects_df.query("leadFunder == 'Innovate UK'").amount.sum()
prop_innovate_uk = (ukri_innovate_uk_funding / ukri_all_projects_funding)

logging.info(f"Proportion of Innovate UK funding: {prop_innovate_uk:.2f}")

2024-07-26 09:42:53,031 - root - INFO - Proportion of Innovate UK funding: 0.20


## Research publications

- Fig 2: Growth and total early-years digital tech publications in 2019-2023
- Fig 4: Proportion of early-years publications associated with digital technologies
- Fig 5: Proportion and growth of major digital tech categories in 2019-2023
- Fig 6: Proportion and growth of major application areas in 2019-2023
- Baseline publication growth across all sectors in 2019-2023
- Geographical distribution


In [19]:
# Load the data
openalex_df = (
    S3.download_obj(
        bucket = S3_BUCKET,
        path_from = S3_OUTPUTS_DIR + 'data/data_openalex.csv',
        download_as='dataframe'
    )
    # Remove the instances tagged with expressive arts due to too much noise
    .query("topics != 'arts'")
)

# Explode by topics
openalex_exploded_df = utils.explode_data(openalex_df).query("topics != 'arts'")

# Get relevant document ids
openalex_tech_ids = get_tech_ids(openalex_exploded_df)
openalex_tech_ids_5y = get_tech_ids_5y(openalex_exploded_df)


In [20]:
logging.info(f"Total number of publications {len(openalex_df)}")

2024-07-25 15:05:31,812 - root - INFO - Total number of publications 69748


### Growth and total early-years digital tech publications

Figure 2

In [21]:
# Filter only technology projects
openalex_tech_type_df = (
    openalex_exploded_df
    .query("id in @openalex_tech_ids")
    .drop_duplicates(['id'])
)

ts_counts_tech = utils.get_timeseries(openalex_tech_type_df, column='id')
utils.plot_quick_ts(ts_counts_tech.query("year >= 2017"), 'counts')

alt.Chart(...)

In [22]:
# Get magnitude and growth
magnitude_growth_openalex = au.ts_magnitude_growth_(
    ts_counts_tech,
    year_start = 2019,
    year_end = 2023
)

growth, magnitude = report_magnitude_growth(magnitude_growth_openalex)

2024-07-25 15:05:31,844 - root - INFO - Growth in 2019-2023: 77.95%
2024-07-25 15:05:31,844 - root - INFO - Total in 2019-2023: 2493.00


### Proportion of early-years publications associated with digital technologies

Figure 4

In [23]:
# Check Technology proportion
openalex_category_distribution_df = utils.get_data_distribution(
    openalex_exploded_df.query("year >= 2019 and year <= 2023"),
    column='type', 
    values=['id']
)

# Get the proportion of technology projects
openalex_prop = openalex_category_distribution_df.query("type == @TECH").counts_prop.iloc[0]
logging.info(f"Proportion of publications: {openalex_prop:.2f}")

# See all major categories
openalex_category_distribution_df

2024-07-25 15:05:31,905 - root - INFO - Proportion of publications: 0.06


,type,counts,counts_prop
0,Biosciences,4330,0.104
1,Child care & preschool,6065,0.146
2,Development & learning,11692,0.281
3,General,23618,0.567
4,Health,20278,0.487
5,Parenting,1353,0.032
6,Society,9569,0.23
7,Technology,2493,0.06


### Digital tech trends in 2019-2023 (publications)

Figure 5

In [24]:
openalex_tech_trends = get_tech_trends(
    openalex_exploded_df, 
    openalex_tech_ids, 
    openalex_tech_ids_5y, 
    ['id']
)

openalex_tech_trends

,subtype,counts,counts_prop,type,magnitude,growth
0,AI,626,0.251,Technology,125.2,91.266376
1,Immersive tech,426,0.171,Technology,85.2,77.456647
2,Internet,1043,0.418,Technology,208.6,123.214286
3,Mobile,732,0.294,Technology,146.4,35.362319


### Application area trends in 2019-2023 (publications)

Figure 6

In [25]:
openalex_application_trends = get_application_trends(
    openalex_exploded_df, 
    openalex_tech_ids, 
    openalex_tech_ids_5y, 
    ['id']
)

openalex_application_trends


,type,counts,counts_prop,magnitude,growth
0,Biosciences,185,0.074,37.0,65.384615
1,Child care & preschool,398,0.16,79.6,96.078431
2,Development & learning,723,0.29,144.6,90.405904
4,Health,906,0.363,181.2,59.349593
5,Parenting,102,0.041,20.4,97.297297
6,Society,429,0.172,85.8,70.930233


Trends information for the heat map

In [26]:
chart_trends.estimate_trend_type(
    openalex_application_trends, 
    magnitude_column='magnitude', 
    growth_column='growth'
)

,type,counts,counts_prop,magnitude,growth,trend_type_suggestion
0,Biosciences,185,0.074,37.0,65.384615,emerging
1,Child care & preschool,398,0.16,79.6,96.078431,emerging*
2,Development & learning,723,0.29,144.6,90.405904,hot
4,Health,906,0.363,181.2,59.349593,hot
5,Parenting,102,0.041,20.4,97.297297,emerging
6,Society,429,0.172,85.8,70.930233,hot*


In [27]:
fig = trends_chart(openalex_application_trends)
fig

alt.Chart(...)

### Geographical distribution

In [28]:
_openalex_countries_df = (
    openalex_exploded_df
    .query('subtype in @tech_subtypes')
    .assign(country_code = lambda df: df.country_code.apply(ast.literal_eval))
    .explode('country_code')
    .drop_duplicates(['id', 'country_code'])
)

n_total_with_codes = len(
    _openalex_countries_df
    .drop_duplicates('id')
    .query("year >= 2019")
    .dropna(subset=['country_code'])
)

openalex_countries_df, _ = utils.get_geographical_distribution(_openalex_countries_df)
openalex_countries_df = (
    openalex_countries_df
    .assign(total = lambda df: df.magnitude*5)
    .assign(proportion = lambda df: df.total / n_total_with_codes)
    .sort_values('proportion', ascending=False)
)
openalex_countries_df.head(10)

,magnitude,growth,country_code,total,proportion
0,113.8,28.102190,US,569.0,0.276616
12,64.0,431.914894,ID,320.0,0.155566
7,38.8,38.202247,GB,194.0,0.094312
5,32.0,15.116279,AU,160.0,0.077783
14,26.0,85.714286,CN,130.0,0.063199
9,19.0,34.000000,CA,95.0,0.046184
4,12.6,182.352941,IN,63.0,0.030627
34,12.0,40.000000,ES,60.0,0.029169
3,11.6,233.333333,DE,58.0,0.028196
8,11.2,90.909091,NL,56.0,0.027224


In [29]:
openalex_US = openalex_countries_df.query("country_code=='US'").proportion.iloc[0]
logging.info(f'US proportion: {openalex_US:.2f}')

openalex_GB = openalex_countries_df.query("country_code=='GB'").proportion.iloc[0]
logging.info(f'GB proportion: {openalex_GB:.2f}')

2024-07-25 15:05:32,609 - root - INFO - US proportion: 0.28
2024-07-25 15:05:32,610 - root - INFO - GB proportion: 0.09


### Baseline growth across all sectors (publications)

In [30]:
# Load the data
openalex_baseline_df = (
    S3.download_obj(
        bucket = S3_BUCKET,
        path_from = S3_OUTPUTS_DIR + 'data/baseline_data_openalex.csv',
        download_as='dataframe'
    )
)

In [31]:
trends_baseline = au.ts_magnitude_growth_(
    ts_df = openalex_baseline_df,
    year_start = 2019,
    year_end = 2023  
)
openalex_baseline_magnitude = trends_baseline.loc['counts'].magnitude
openalex_baseline_growth = trends_baseline.loc['counts'].growth
logging.info(f"Publication baseline growth: {openalex_baseline_growth:.2f}%")

2024-07-25 15:05:32,711 - root - INFO - Publication baseline growth: -3.55%


### Baseline growth for edtech (publications)

In [32]:
# Load the data
openalex_baseline_df_edtech = (
    S3.download_obj(
        bucket = S3_BUCKET,
        path_from = S3_OUTPUTS_DIR + 'data/baseline_data_openalex_edtech.csv',
        download_as='dataframe'
    )
)

trends_baseline = au.ts_magnitude_growth_(
    ts_df = openalex_baseline_df_edtech,
    year_start = 2019,
    year_end = 2023  
)
openalex_baseline_magnitude_edtech = trends_baseline.loc['counts'].magnitude*5
openalex_baseline_growth_edtech = trends_baseline.loc['counts'].growth
logging.info(f"Edtech publications in total: {openalex_baseline_magnitude_edtech:.2f}")
logging.info(f"Edtech publication baseline growth: {openalex_baseline_growth_edtech:.2f}%")

2024-07-25 15:05:32,804 - root - INFO - Edtech publications in total: 29378.00
2024-07-25 15:05:32,805 - root - INFO - Edtech publication baseline growth: 49.10%


## Patents

- Fig 2: Growth and total early-years digital tech patents in 2019-2023
- Fig 4: Proportion of early-years patents associated with digital technologies
- Fig 5: Proportion and growth of major digital tech categories in 2019-2023
- Fig 6: Proportion and growth of major application areas in 2019-2023
- Baseline patent growth across all sectors in 2019-2023
- Geographical distribution


In [33]:
# Load the data
patents_df = (
    S3.download_obj(
        bucket = S3_BUCKET,
        path_from = S3_OUTPUTS_DIR + 'data/data_patents.csv',
        download_as='dataframe'
    )
    # Remove the instances tagged with expressive arts due to too much noise
    .query("topics != 'arts'")
)

# Explode by topics
patents_exploded_df = utils.explode_data(patents_df).query("topics != 'arts'")

# Get relevant document ids
patents_tech_ids = get_tech_ids(patents_exploded_df)
patents_tech_ids_5y = get_tech_ids_5y(patents_exploded_df)

### Growth and total early-years digital tech patents

Figure 2

In [34]:
# Filter only technology projects
patents_tech_type_df = (
    patents_exploded_df
    .query("id in @patents_tech_ids")
    .drop_duplicates(['id'])
)

ts_counts_tech = utils.get_timeseries(patents_tech_type_df, column='id')
utils.plot_quick_ts(ts_counts_tech, 'counts')

alt.Chart(...)

In [35]:
# Get magnitude and growth
magnitude_growth_patents= au.ts_magnitude_growth_(
    ts_counts_tech,
    year_start = 2019,
    year_end = 2023
)

growth, magnitude = report_magnitude_growth(magnitude_growth_patents)

2024-07-25 15:05:35,831 - root - INFO - Growth in 2019-2023: -5.22%
2024-07-25 15:05:35,832 - root - INFO - Total in 2019-2023: 2618.00


Check stats without including China

In [36]:
# Filter only technology projects
patents_tech_type_df_wout_CN = (
    patents_exploded_df
    .query('country_code != "CN"')
    .query("id in @patents_tech_ids")
    .drop_duplicates(['id'])
)

ts_counts_tech_wout_CN = utils.get_timeseries(patents_tech_type_df_wout_CN, column='id')
utils.plot_quick_ts(ts_counts_tech_wout_CN, 'counts')

alt.Chart(...)

In [37]:
# Get magnitude and growth
magnitude_growth_patents_wout_CN = au.ts_magnitude_growth_(
    ts_counts_tech_wout_CN,
    year_start = 2019,
    year_end = 2023
)

growth, magnitude = report_magnitude_growth(magnitude_growth_patents_wout_CN)

2024-07-25 15:05:35,858 - root - INFO - Growth in 2019-2023: 20.00%
2024-07-25 15:05:35,859 - root - INFO - Total in 2019-2023: 945.00


### Proportion of early-years patents associated with digital technologies

Figure 4

In [38]:
# Check Technology proportion
patents_category_distribution_df = utils.get_data_distribution(
    patents_exploded_df.query("year >= 2019 and year <= 2023"),
    column='type', 
    values=['id']
)

# Get the proportion of technology projects
patents_prop = patents_category_distribution_df.query("type == @TECH").counts_prop.iloc[0]
logging.info(f"Proportion of technology patents: {patents_prop:.2f}")

# See all major categories
patents_category_distribution_df

2024-07-25 15:05:35,881 - root - INFO - Proportion of technology patents: 0.18


,type,counts,counts_prop
0,Biosciences,275,0.019
1,Child care & preschool,619,0.043
2,Development & learning,767,0.053
3,General,10587,0.737
4,Health,3490,0.243
5,Parenting,277,0.019
6,Society,31,0.002
7,Technology,2618,0.182


### Digital tech trends in 2019-2023 (patents)

Figure 5

In [39]:
patents_tech_trends = get_tech_trends(
    patents_exploded_df, 
    patents_tech_ids, 
    patents_tech_ids_5y, 
    ['id']
)

patents_tech_trends

,subtype,counts,counts_prop,type,magnitude,growth
0,AI,1637,0.625,Technology,327.4,9.630459
1,Immersive tech,874,0.334,Technology,174.8,-14.874552
2,Internet,10,0.004,Technology,2.0,-33.333333
3,Mobile,677,0.259,Technology,135.4,-42.229730


### Application area trends in 2019-2023 (patents)

Figure 6

In [40]:
patent_application_trends = get_application_trends(
    patents_exploded_df, 
    patents_tech_ids, 
    patents_tech_ids_5y, 
    ['id']
)

patent_application_trends

,type,counts,counts_prop,magnitude,growth
0,Biosciences,85,0.032,17.0,177.272727
1,Child care & preschool,173,0.066,34.6,-23.622047
2,Development & learning,260,0.099,52.0,0.714286
4,Health,920,0.351,184.0,5.482042
5,Parenting,261,0.1,52.2,-28.491620
6,Society,2,0.001,0.4,inf


Trends information for the heat map

In [41]:
chart_trends.estimate_trend_type(
    patent_application_trends, 
    magnitude_column='magnitude', 
    growth_column='growth'
)

,type,counts,counts_prop,magnitude,growth,trend_type_suggestion
0,Biosciences,85,0.032,17.0,177.272727,emerging
1,Child care & preschool,173,0.066,34.6,-23.622047,dormant
2,Development & learning,260,0.099,52.0,0.714286,hot
4,Health,920,0.351,184.0,5.482042,hot
5,Parenting,261,0.1,52.2,-28.491620,stable
6,Society,2,0.001,0.4,inf,emerging


- Society has such a tiny number of patents we'll designate as 'dormant' instead
- Development and learning had a growth rate very close to 0, so not that strongly emerging - more between dormant and emerging.

In [42]:
fig = trends_chart(patent_application_trends)
fig

alt.Chart(...)

### Geographical distribution

In [43]:
_patents_countries_df = (
    patents_exploded_df
    .query('subtype in @tech_subtypes')
)

n_total_with_codes = len(
    _patents_countries_df
    .drop_duplicates('id')
    .query("year >= 2019")
    .dropna(subset=['country_code'])
)

patents_countries_df, _ = utils.get_geographical_distribution(_patents_countries_df)
patents_countries_df = (
    patents_countries_df
    .assign(proportion = lambda df: df.magnitude*5 / n_total_with_codes)
    .sort_values('proportion', ascending=False)
)
patents_countries_df.head(15)

,magnitude,growth,country_code,proportion
0,334.6,-15.964126,CN,0.639037
1,80.0,32.596685,KR,0.152788
2,36.6,-13.274336,US,0.069901
6,20.8,-9.836066,WO,0.039725
11,12.2,11.428571,JP,0.023300
4,7.8,141.666667,EP,0.014897
5,6.4,69.230769,TW,0.012223
3,4.2,800.000000,AU,0.008021
9,2.8,120.000000,TR,0.005348
7,2.8,125.000000,CA,0.005348


In [44]:
openalex_US = patents_countries_df.query("country_code=='CN'").proportion.iloc[0]
logging.info(f'CN proportion: {openalex_US:.2f}')

openalex_US = patents_countries_df.query("country_code=='US'").proportion.iloc[0]
logging.info(f'US proportion: {openalex_US:.2f}')

openalex_GB = patents_countries_df.query("country_code=='GB'").proportion.iloc[0]
logging.info(f'GB proportion: {openalex_GB:.2f}')

2024-07-25 15:05:36,310 - root - INFO - CN proportion: 0.64
2024-07-25 15:05:36,312 - root - INFO - US proportion: 0.07
2024-07-25 15:05:36,314 - root - INFO - GB proportion: 0.00


### Baseline growth across all sectors 

In [45]:
# Load the data
patents_baseline_df = (
    S3.download_obj(
        bucket = S3_BUCKET,
        path_from = S3_OUTPUTS_DIR + 'data/baseline_data_patents.csv',
        download_as='dataframe'
    )
)

In [46]:
trends_baseline = au.ts_magnitude_growth_(
    ts_df = patents_baseline_df,
    year_start = 2019,
    year_end = 2023  
)
patents_baseline_magnitude = trends_baseline.loc['counts'].magnitude
patents_baseline_growth = trends_baseline.loc['counts'].growth
logging.info(f"UKRI baseline growth: {patents_baseline_growth:.2f}%")

2024-07-25 15:05:36,435 - root - INFO - UKRI baseline growth: 31.19%


## Venture funding

- Fig 2: Growth and total early-years digital tech funding in 2019-2023
- Fig 3: Venture funding by deal size over years
- Fig 4: Proportion of early-years funding associated with digital technologies
- Fig 5: Proportion and growth of major digital tech funding in 2019-2023
- Fig 6: Proportion and growth of major application areas in 2019-2023
- Baseline funding growth across all sectors in 2019-2023
- Geographical distribution


In [47]:
# Load the data
crunchbase_df = (
    S3.download_obj(
        bucket = S3_BUCKET,
        path_from = S3_OUTPUTS_DIR + 'data/data_crunchbase.csv',
        download_as='dataframe'
    )
    # Remove the instances tagged with expressive arts due to too much noise
    .query("topics != 'arts'")
)

# Explode by topics
crunchbase_exploded_df = utils.explode_data(crunchbase_df, is_crunchbase=True).query("topics != 'arts'")

# Get relevant document ids
crunchbase_tech_ids = get_tech_ids(crunchbase_exploded_df)
crunchbase_tech_ids_5y = get_tech_ids_5y(crunchbase_exploded_df)


In [48]:
logging.info(f"Total number of venture funding rounds {len(crunchbase_df)}")

2024-07-25 15:05:36,898 - root - INFO - Total number of venture funding rounds 2540


### Growth and total early-years digital tech investment

Figure 2

In [49]:
# Filter only technology projects
crunchbase_tech_type_df = (
    crunchbase_exploded_df
    .query("id in @crunchbase_tech_ids")
    .drop_duplicates(['id'])
)

ts_amounts_tech = utils.get_timeseries(crunchbase_tech_type_df, column='amount')
ts_counts_tech = utils.get_timeseries(crunchbase_tech_type_df, column='id')
utils.plot_quick_ts(ts_amounts_tech, 'amount')

alt.Chart(...)

In [50]:
# Get magnitude and growth
magnitude_growth_crunchbase = au.ts_magnitude_growth_(
    ts_amounts_tech,
    year_start = 2019,
    year_end = 2023
)

growth, magnitude = report_magnitude_growth(magnitude_growth_crunchbase)

2024-07-25 15:05:36,926 - root - INFO - Growth in 2019-2023: 41.20%
2024-07-25 15:05:36,926 - root - INFO - Total in 2019-2023: 4232509.55


Check growth if removing large deals (larger than £100M)

In [51]:
# Filter only technology projects
crunchbase_tech_type_df_wout_large = (
    crunchbase_exploded_df
    .query("amount < 100000")
    .query("id in @crunchbase_tech_ids")
    .drop_duplicates(['id'])
)

ts_amounts_tech = utils.get_timeseries(crunchbase_tech_type_df_wout_large, column='amount')
ts_counts_tech = utils.get_timeseries(crunchbase_tech_type_df_wout_large, column='id')
utils.plot_quick_ts(ts_amounts_tech, 'amount')

alt.Chart(...)

In [52]:
# Get magnitude and growth
magnitude_growth_crunchbase = au.ts_magnitude_growth_(
    ts_amounts_tech,
    year_start = 2019,
    year_end = 2023
)

growth, magnitude = report_magnitude_growth(magnitude_growth_crunchbase)

2024-07-25 15:05:36,952 - root - INFO - Growth in 2019-2023: 3.26%
2024-07-25 15:05:36,953 - root - INFO - Total in 2019-2023: 2591931.87


### Venture funding by deal size over years

Figure 3

In [53]:
deal_order = ["n/a", "£0-5M", "£5-20M", "£20-100M", "£100M+"]

def deal_amount_to_range_coarse(
    amount: float, currency: str = "£", categories: bool = True
) -> str:
    """
    Convert amounts to range in millions
    Args:
        amount: Investment amount (in GBP thousands)
        categories: If True, adding indicative deal categories
        currency: Currency symbol
    """
    amount /= 1e3
    if (amount >= 0.001) and (amount <= 5):
        return f"{currency}0-5M" if not categories else f"{currency}0-5M"
    elif (amount > 5) and (amount <= 20):
        return f"{currency}5-20M" if not categories else f"{currency}5-20M"
    elif (amount > 20) and (amount <= 100):
        return f"{currency}20-100M" if not categories else f"{currency}20-100M"
    elif amount > 100:
        return f"{currency}100M+"
    else:
        return "n/a"

In [54]:
funding_df_ranges = (
    crunchbase_df
    .query("id in @crunchbase_tech_ids")
    .assign(_amount = lambda df: df.amount/1000)
    .assign(deal_type=lambda df: df.amount.apply(deal_amount_to_range_coarse))
    .astype({"deal_type": "category"})
    .assign(deal_type=lambda x: x.deal_type.cat.set_categories(deal_order))
    .drop_duplicates('id')
)

deal_data = (
    funding_df_ranges.groupby(["year", "deal_type"], as_index=True)
    .agg(
        counts=("id", "count"),
        total_amount=("amount", "sum"),
    )
    .reset_index()
    .query("year >= 2013")
    .query("year <= 2023")
    .assign(total_amount=lambda df: df.total_amount / 1000)
)
deal_data_wide_df = (
    deal_data.pivot(index="year", columns="deal_type", values="total_amount")
    .fillna(0)
    .astype(int)
    .reset_index()
)

deal_data_wide_df = (
    deal_data_wide_df
    .merge(deal_data.groupby('year').total_amount.sum().reset_index(), on='year', how='left')
)

deal_data_wide_df

,year,n/a,£0-5M,£5-20M,£20-100M,£100M+,total_amount
0,2013,0,55,46,22,0,124.429639
1,2014,0,80,88,174,0,343.423021
2,2015,0,95,89,424,0,609.399647
3,2016,0,101,158,307,642,1209.904105
4,2017,0,109,179,40,0,330.011927
5,2018,0,121,201,218,114,656.101867
6,2019,0,109,156,236,348,851.677676
7,2020,0,90,123,456,115,785.950728
8,2021,0,110,170,564,1175,2021.634749
9,2022,0,85,163,21,0,269.569469


In [55]:
deal_data_wide_counts_df = (
    deal_data.pivot(index="year", columns="deal_type", values="counts")
    .fillna(0)
    .astype(int)
    .reset_index()
)
deal_data_wide_counts_df

deal_type,year,n/a,£0-5M,£5-20M,£20-100M,£100M+
0,2013,0,102,5,1,0
1,2014,0,120,9,5,0
2,2015,0,126,9,8,0
3,2016,0,121,16,7,3
4,2017,0,126,18,2,0
5,2018,0,114,18,6,1
6,2019,0,104,18,6,3
7,2020,0,103,13,8,1
8,2021,1,100,17,13,5
9,2022,0,62,14,1,0


In [56]:
# Sense check
deal_data_wide_df.query("year >= 2019 and year <= 2023").total_amount.sum()

4232.509547607872

### Proportion of early-years investment associated with digital technologies

Figure 4

In [57]:
# Check Technology proportion using another approach
crunchbase_category_distribution_df = utils.get_data_distribution(
    crunchbase_exploded_df.query("year >= 2019 and year <= 2023"),
    column='type', 
    values=['id', 'amount']
)

# Get the proportion of technology projects
crunchbase_prop = crunchbase_category_distribution_df.query("type == @TECH").amount_prop.iloc[0]
logging.info(f"Proportion of investment: {crunchbase_prop:.2f}")

# See all major categories
crunchbase_category_distribution_df

2024-07-25 15:05:37,005 - root - INFO - Proportion of investment: 0.56


,type,counts,counts_prop,amount,amount_prop
0,Biosciences,23,0.021,55364.986316,0.007
1,Child care & preschool,127,0.118,610225.151123,0.081
2,Development & learning,232,0.215,2292446.085317,0.304
3,General,420,0.39,2695685.632986,0.357
4,Health,425,0.394,3026034.566599,0.401
5,Parenting,177,0.164,720381.014435,0.095
6,Society,108,0.1,509445.801672,0.067
7,Technology,513,0.476,4232509.547608,0.561


### Digital tech trends in 2019-2023 (investment)

Figure 5

In [58]:
crunchbase_tech_trends = get_tech_trends(
    crunchbase_exploded_df, 
    crunchbase_tech_ids, 
    crunchbase_tech_ids_5y, 
    ['amount', 'id']
)

crunchbase_tech_trends

,subtype,amount,amount_prop,counts,counts_prop,type,magnitude,growth
0,AI,625016.449792,0.148,173,0.337,Technology,125003.289958,73.901417
1,Immersive tech,284661.474141,0.067,62,0.121,Technology,56932.294828,-17.969141
2,Internet,2892628.722755,0.683,110,0.214,Technology,578525.744551,41.941131
3,Mobile,1377192.183934,0.325,234,0.456,Technology,275438.436787,13.107150
4,Operations,435877.562785,0.103,89,0.173,Child care & preschool,87175.512557,8.858934


### Application area trends in 2019-2023 (investment)

Figure 6

In [59]:
crunchbase_application_trends = get_application_trends(
    crunchbase_exploded_df, 
    crunchbase_tech_ids, 
    crunchbase_tech_ids_5y, 
    ['amount', 'id']
)

crunchbase_application_trends


,type,amount,amount_prop,counts,counts_prop,magnitude,growth
0,Biosciences,9026.586836,0.002,9,0.018,1805.317367,1748.296224
1,Child care & preschool,463498.943387,0.11,102,0.199,92699.788677,-0.863038
2,Development & learning,1459553.275091,0.345,147,0.287,291910.655018,44.765309
4,Health,620837.404541,0.147,144,0.281,124167.480908,40.466729
5,Parenting,273485.631675,0.065,105,0.205,54697.126335,-7.308249
6,Society,165733.006945,0.039,50,0.097,33146.601389,22.534738


Trends information for the heat map

In [60]:
chart_trends.estimate_trend_type(
    crunchbase_application_trends, 
    magnitude_column='magnitude', 
    growth_column='growth'
)[['type', 'magnitude', 'growth', 'trend_type_suggestion']]

,type,magnitude,growth,trend_type_suggestion
0,Biosciences,1805.317367,1748.296224,emerging
1,Child care & preschool,92699.788677,-0.863038,stable
2,Development & learning,291910.655018,44.765309,hot
4,Health,124167.480908,40.466729,hot
5,Parenting,54697.126335,-7.308249,dormant
6,Society,33146.601389,22.534738,emerging


In [61]:
fig = trends_chart(
    crunchbase_application_trends
    .query("type != 'Biosciences'")
)
fig

alt.Chart(...)

### Geographical distribution

In [62]:
_crunchbase_countries_df = (
    crunchbase_exploded_df
    .query('id in @crunchbase_tech_ids')
)

n_total_with_codes = (
    _crunchbase_countries_df
    .query("year >= 2019")
    .drop_duplicates('id')
    .dropna(subset=['country_code'])
    .amount.sum()
)

crunchbase_countries_df, _ = utils.get_geographical_distribution(
    _crunchbase_countries_df,
    column = 'amount',
)
crunchbase_countries_df = (
    crunchbase_countries_df
    .assign(total = lambda df: df.magnitude*5)
    .assign(proportion = lambda df: df.total / n_total_with_codes)
    .sort_values('proportion', ascending=False)
)
crunchbase_countries_df.head(10)

,magnitude,growth,country_code,total,proportion
3,523630.007604,147.721838,USA,2.618150e+06,0.618581
11,158982.268980,13.455687,IND,7.949113e+05,0.187811
1,46772.956806,-81.965750,CHN,2.338648e+05,0.055254
8,37109.720306,-76.741053,GBR,1.855486e+05,0.043839
4,11868.538098,602.395266,CAN,5.934269e+04,0.014021
31,10437.474855,-23.922810,JPN,5.218737e+04,0.012330
35,9495.637346,1626.850918,KOR,4.747819e+04,0.011218
6,7872.595920,179.978612,ESP,3.936298e+04,0.009300
5,7784.813000,135.579853,DEU,3.892407e+04,0.009196
0,6010.114111,235.287954,FRA,3.005057e+04,0.007100


In [63]:
# Double check GBR total investment
(
    crunchbase_exploded_df
    .query("id in @crunchbase_tech_ids_5y")
    .query("country_code == 'GBR'")
    .drop_duplicates('id')
    .amount.sum()
)

185548.6015294242

In [64]:
# Double check investment types
sorted(list(crunchbase_exploded_df.investment_type.unique()))

['angel',
 'convertible_note',
 'equity_crowdfunding',
 'non_equity_assistance',
 'pre_seed',
 'product_crowdfunding',
 'secondary_market',
 'seed',
 'series_a',
 'series_b',
 'series_c',
 'series_d',
 'series_e',
 'series_unknown']

### Baseline growth across all sectors (venture funding)

In [65]:
# Load the data
crunchbase_baseline_df = (
    S3.download_obj(
        bucket = S3_BUCKET,
        path_from = S3_OUTPUTS_DIR + 'data/baseline_data_crunchbase.csv',
        download_as='dataframe'
    )
)

In [66]:
trends_baseline = au.ts_magnitude_growth_(
    ts_df = crunchbase_baseline_df,
    year_start = 2019,
    year_end = 2023  
)
crunchbase_baseline_magnitude = trends_baseline.loc['amount'].magnitude
crunchbase_baseline_growth = trends_baseline.loc['amount'].growth
logging.info(f"Investment baseline growth: {crunchbase_baseline_growth:.2f}%")

2024-07-25 15:05:37,551 - root - INFO - Investment baseline growth: 81.75%


### Baseline growth across all sectors (edtech)

In [67]:
# Load the data
crunchbase_baseline_df_edtech = (
    S3.download_obj(
        bucket = S3_BUCKET,
        path_from = S3_OUTPUTS_DIR + 'data/baseline_data_crunchbase_edtech.csv',
        download_as='dataframe'
    )
)

trends_baseline = au.ts_magnitude_growth_(
    ts_df = crunchbase_baseline_df_edtech,
    year_start = 2019,
    year_end = 2023  
)
crunchbase_baseline_magnitude_edtech = trends_baseline.loc['amount'].magnitude*5
crunchbase_baseline_growth_edtech = trends_baseline.loc['amount'].growth
logging.info(f"Edtech investment baseline growth: {crunchbase_baseline_growth_edtech:.2f}%")
logging.info(f"Edtech magnitude: {crunchbase_baseline_magnitude_edtech:.2f}")

2024-07-25 15:05:37,657 - root - INFO - Edtech investment baseline growth: 34.44%
2024-07-25 15:05:37,658 - root - INFO - Edtech magnitude: 32859079.79
